# 02 — Short Interest Signal Research

This notebook constructs and evaluates the core short-selling signals from FINRA data.  The primary question: does short interest / short flow contain information about subsequent cross-sectional returns?

### Literature context
* Diether, Malloy & Scherbina (2002): stocks with high disagreement (proxied by SI) underperform
* Asquith, Pathak & Ritter (2005): high SI predicts negative returns, especially in small/illiquid stocks
* Engelberg, Reed & Ringgenberg (2012): daily short volume (SVR) predicts returns at short horizons

We test these hypotheses on our universe, applying Benjamini-Hochberg correction for multiple testing.

In [ ]:
import sys; sys.path.insert(0, "../src")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from securities_lending.analysis import ICAnalyzer
from securities_lending.viz import plot_ic_tearsheet

plt.rcParams.update({"figure.facecolor": "white", "axes.grid": True, "grid.alpha": 0.3})

# Load the pre-built feature panel
features = pd.read_parquet("../data/processed/features.parquet")
features["date"] = pd.to_datetime(features["date"])
print(f"Features: {features.shape}")
print("Columns:", features.columns.tolist())

## 1. Signal Distributions

Before computing IC, we examine the cross-sectional distribution of each signal to check for extreme skewness or outliers that would distort the Spearman correlation.

In [ ]:
SIGNALS = ["svr_z20", "svr_trend5", "si_pct_float", "si_dtc",
            "short_pressure", "borrow_rate_bps"]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flat, SIGNALS):
    if col not in features.columns:
        ax.set_title(f"{col} (N/A)"); continue
    # Pool all cross-sections
    data = features[col].dropna()
    ax.hist(data, bins=60, color="#2B6CB0", alpha=0.8, edgecolor="white")
    ax.axvline(data.mean(), color="#C53030", linewidth=1.5,
               label=f"mean={data.mean():.3f}")
    ax.set_title(col)
    ax.legend(fontsize=8)
fig.suptitle("Cross-Sectional Signal Distributions (pooled)", fontsize=12)
plt.tight_layout()
plt.show()

## 2. Information Coefficient (IC) Analysis

We compute the cross-sectional Spearman rank correlation between each signal measured at time *t* and the forward return over horizons {1, 5, 10, 21} days.  Cross-sections are winsorized at 1%/99% before computing IC.

In [ ]:
def pivot(df, col):
    return df.pivot(index="date", columns="symbol", values=col)

ret5_panel = pivot(features, "ret_fwd_5d")
HORIZONS = [1, 5, 10, 21]

# Use svr_z20 as the primary signal for the full tear sheet
if "svr_z20" in features.columns:
    svr_panel = pivot(features, "svr_z20")
    analyzer = ICAnalyzer(signal_panel=svr_panel, return_panel=ret5_panel)
    svr_ic = analyzer.run(horizons=HORIZONS, signal_name="svr_z20")
    fig = plot_ic_tearsheet(svr_ic, signal_name="SVR z-score (svr_z20)")
    plt.show()
    print("
SVR IC Summary:")
    for h, r in svr_ic.items():
        print(r)

## 3. Multi-Signal IC Table with FDR Correction

With 6 signals × 4 horizons = 24 tests, we apply Benjamini-Hochberg FDR correction.  Only signals that pass at q=0.05 are considered candidates for portfolio construction.

In [ ]:
signal_panels = {}
for col in SIGNALS:
    if col in features.columns:
        p = pivot(features, col).dropna(how="all", axis=1).dropna(how="all", axis=0)
        if not p.empty:
            signal_panels[col] = p

# Use the first signal panel to set up the analyzer object
if signal_panels:
    first_name, first_panel = next(iter(signal_panels.items()))
    analyzer_multi = ICAnalyzer(signal_panel=first_panel, return_panel=ret5_panel)
    ic_table = analyzer_multi.run_multiple(signal_panels, horizons=HORIZONS)
    print(ic_table.to_string(index=False))
    # Show only significant results
    print("
Significant signals (BH q<0.05):")
    print(ic_table[ic_table["significant_bh"]].to_string(index=False))

## 4. IC Decay — How Quickly Does the Signal Decay?

A signal with IC that persists for 21+ days has a longer predictive horizon than one that decays after 5 days.  The decay pattern informs the optimal rebalancing frequency.

In [ ]:
if signal_panels:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    colors = ["#2B6CB0", "#C53030", "#276749", "#D69E2E", "#553C9A", "#2C7A7B"]
    for (name, _), color in zip(signal_panels.items(), colors):
        ics = []
        for h in HORIZONS:
            subset = ic_table[(ic_table["signal"] == name) & (ic_table["horizon"] == h)]
            if not subset.empty:
                ics.append(subset["mean_ic"].values[0])
            else:
                ics.append(np.nan)
        axes[0].plot(HORIZONS, ics, "o-", color=color, linewidth=1.5,
                     markersize=5, label=name)
    axes[0].axhline(0, color="black", linewidth=0.8)
    axes[0].set_title("IC Decay Curves")
    axes[0].set_xlabel("Forward Horizon (days)")
    axes[0].set_ylabel("Mean IC")
    axes[0].legend(fontsize=8)

    # ICIR decay
    for (name, _), color in zip(signal_panels.items(), colors):
        icirs = []
        for h in HORIZONS:
            subset = ic_table[(ic_table["signal"] == name) & (ic_table["horizon"] == h)]
            if not subset.empty:
                icirs.append(subset["icir"].values[0])
            else:
                icirs.append(np.nan)
        axes[1].plot(HORIZONS, icirs, "o-", color=color, linewidth=1.5,
                     markersize=5, label=name)
    axes[1].axhline(0, color="black", linewidth=0.8)
    axes[1].axhline(0.5, color="grey", linewidth=0.8, linestyle="--",
                    label="ICIR=0.5 threshold")
    axes[1].set_title("ICIR Decay Curves")
    axes[1].set_xlabel("Forward Horizon (days)")
    axes[1].set_ylabel("ICIR")
    axes[1].legend(fontsize=8)
    fig.tight_layout()
    plt.show()

## 5. Portfolio Quintile Sort

We sort stocks into 5 quantiles on the primary signal (svr_z20) and compute equal-weighted portfolio returns over a 5-day holding period.

**Key tests:**
* Is the Q5−Q1 spread statistically significant (t > 2)?
* Is the spread monotone across quintiles?
* Does it survive transaction costs?

In [ ]:
from securities_lending.analysis import PortfolioSorter
from securities_lending.viz import plot_portfolio_quintiles

borrow_panel = pivot(features, "borrow_rate_bps") if "borrow_rate_bps" in features.columns else None

if "svr_z20" in signal_panels:
    sorter = PortfolioSorter(
        signal_panel=signal_panels["svr_z20"],
        return_panel=ret5_panel,
        borrow_rate_panel=borrow_panel,
        n_quantiles=5,
        holding_period=5,
    )
    result = sorter.run(signal_name="SVR z-score", cost_scenarios=[5, 10, 20])
    fig = plot_portfolio_quintiles(result)
    plt.show()
    print(f"
L/S spread: {result.ls_spread_ann*100:.2f}%/yr")
    print(f"L/S Sharpe: {result.ls_sharpe:.2f}")
    print(f"t-stat: {result.ls_t_stat:.2f}")
    print(f"Monotonicity: {result.monotonicity:.0%}")
    print("
Net spreads by TC scenario:")
    for tc, net in result.net_spreads.items():
        print(f"  {tc}bps one-way: {net*100:.2f}%/yr")